In [5]:
# ODIN EXECUTIVE SUITE v4.2 (Emoji-Free Professional Edition)
# Professional Business Intelligence with Robust PDF Rendering & Full Data Tables

import os
import re
import xmlrpc.client
import pandas as pd
import numpy as np
import json
from typing import Dict, Any, Optional, List, TypedDict
from datetime import datetime, timedelta
from dataclasses import dataclass

# Environment
from dotenv import load_dotenv

# LangGraph & AI
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, 
    TableStyle, PageBreak, Image
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT, TA_JUSTIFY

# ================================================================
# 1. CONFIGURATION
# ================================================================
load_dotenv()

class Config:
    # Odoo Configuration
    ODOO_URL = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
    ODOO_DB = os.getenv('ODOO_DB', 'vendyltd')
    ODOO_USER = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
    ODOO_PASSWORD = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')
    
    # AI Configuration
    LLM_MODEL = "gpt-4-turbo"
    
    # Output
    REPORT_DIR = "./odin_reports"
    
    @classmethod
    def ensure_dirs(cls):
        os.makedirs(cls.REPORT_DIR, exist_ok=True)

Config.ensure_dirs()

# ================================================================
# 2. DATA MODELS
# ================================================================
@dataclass
class AgentResult:
    data: Dict[str, Any]
    timestamp: datetime
    success: bool
    error: Optional[str] = None

@dataclass
class BusinessMetrics:
    sales: Dict[str, Any]
    delivery: Dict[str, Any]
    inventory: Dict[str, Any]
    accounting: Dict[str, Any]
    cfo: Dict[str, Any]
    timestamp: datetime

# State Schema for LangGraph
class GraphState(TypedDict):
    date_from: Optional[str]
    date_to: Optional[str]
    metrics: Optional[BusinessMetrics]
    contradictions: Optional[Dict]
    ceo_brief: Optional[Dict]

# ================================================================
# 3. CORE ODOO CLIENT
# ================================================================
class OdooClient:
    def __init__(self):
        self._cache = {}
        self.connect()
    
    def connect(self):
        try:
            common = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/common")
            self.uid = common.authenticate(Config.ODOO_DB, Config.ODOO_USER, Config.ODOO_PASSWORD, {})
            self.models = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/object")
            self.connected = bool(self.uid)
            print("✅ Connected to Odoo" if self.connected else "❌ Authentication failed")
        except Exception as e:
            print(f"❌ Odoo connection failed: {e}")
            self.connected = False
    
    def read(self, model: str, domain: Optional[List] = None, fields: Optional[List] = None) -> List[Dict]:
        if not self.connected: return []
        try:
            return self.models.execute_kw(
                Config.ODOO_DB, self.uid, Config.ODOO_PASSWORD,
                model, 'search_read', [domain or []], {'fields': fields or []}
            )
        except Exception as e:
            print(f"⚠️ Error reading {model}: {e}")
            return []

odoo = OdooClient()

# ================================================================
# 4. SAFETY UTILITIES
# ================================================================
class SafetyUtils:
    @staticmethod
    def safe_df(records: List[Dict], required_cols: List[str]) -> pd.DataFrame:
        if not records: return pd.DataFrame(columns=required_cols)
        try:
            df = pd.DataFrame(records)
            for col in required_cols:
                if col not in df.columns: df[col] = 0
            return df
        except Exception:
            return pd.DataFrame(columns=required_cols)
    
    @staticmethod
    def calculate_percentage(numerator, denominator, default=0.0):
        if denominator == 0: return default
        return (numerator / denominator) * 100

# ================================================================
# 5. AGENTS (Operational & Analytics)
# ================================================================
class SalesAgent:
    def execute(self, date_from=None, date_to=None) -> AgentResult:
        try:
            domain = []
            if date_from: domain.append(('date_order', '>=', date_from))
            if date_to: domain.append(('date_order', '<=', date_to))
            
            orders = odoo.read('sale.order', domain, ['amount_total', 'state'])
            df = SafetyUtils.safe_df(orders, ['amount_total', 'state'])
            
            if len(df) == 0:
                data = {"total_sales": 0, "order_count": 0, "conversion_rate": 0, "average_order_value": 0, "draft_orders": 0}
            else:
                df['amount_total'] = pd.to_numeric(df['amount_total'], errors='coerce').fillna(0)
                total = df['amount_total'].sum()
                count = len(df)
                data = {
                    "total_sales": round(total, 2),
                    "order_count": count,
                    "average_order_value": round(total / count if count > 0 else 0, 2),
                    "conversion_rate": round(SafetyUtils.calculate_percentage(
                        df['state'].isin(['sale', 'done']).sum(), count), 2),
                    "draft_orders": int((df['state'] == 'draft').sum())
                }
            return AgentResult(data=data, timestamp=datetime.now(), success=True)
        except Exception as e:
            return AgentResult(data={}, timestamp=datetime.now(), success=False, error=str(e))

class DeliveryAgent:
    def execute(self, date_from=None, date_to=None) -> AgentResult:
        try:
            domain = []
            if date_from: domain.append(('scheduled_date', '>=', date_from))
            if date_to: domain.append(('scheduled_date', '<=', date_to))
            
            pickings = odoo.read('stock.picking', domain, ['state'])
            df = SafetyUtils.safe_df(pickings, ['state'])
            
            if len(df) == 0:
                data = {"completion_rate": 0, "total_deliveries": 0, "pending": 0, "completed": 0, "efficiency_score": 0}
            else:
                completed = int((df['state'] == 'done').sum())
                total = len(df)
                rate = SafetyUtils.calculate_percentage(completed, total)
                data = {
                    "total_deliveries": total,
                    "completed": completed,
                    "pending": total - completed - int((df['state'] == 'cancel').sum()),
                    "completion_rate": round(rate, 2),
                    "efficiency_score": round(max(0, min(100, rate * 0.8)), 2)
                }
            return AgentResult(data=data, timestamp=datetime.now(), success=True)
        except Exception as e:
            return AgentResult(data={}, timestamp=datetime.now(), success=False, error=str(e))

class InventoryAgent:
    def execute(self) -> AgentResult:
        try:
            quants = odoo.read('stock.quant', None, ['quantity', 'reserved_quantity'])
            df = SafetyUtils.safe_df(quants, ['quantity', 'reserved_quantity'])
            
            if len(df) == 0:
                data = {"available_units": 0, "zero_negative_items": 0, "stockout_risk_score": 100, "health_status": "CRITICAL"}
            else:
                df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0)
                total = df['quantity'].sum()
                zero_neg = int((df['quantity'] <= 0).sum())
                risk = min(100, (zero_neg / len(df)) * 100 * 0.7)
                
                health = "CRITICAL" if risk > 70 else "HIGH_RISK" if risk > 40 else "HEALTHY"
                data = {
                    "total_units": float(total),
                    "available_units": float(total - df['reserved_quantity'].sum()),
                    "zero_negative_items": zero_neg,
                    "stockout_risk_score": round(risk, 2),
                    "health_status": health
                }
            return AgentResult(data=data, timestamp=datetime.now(), success=True)
        except Exception as e:
            return AgentResult(data={}, timestamp=datetime.now(), success=False, error=str(e))

class AccountingAgent:
    def execute(self, date_from=None, date_to=None) -> AgentResult:
        try:
            domain = [('state', '=', 'posted')]
            if date_from: domain.append(('date', '>=', date_from))
            if date_to: domain.append(('date', '<=', date_to))
            
            moves = odoo.read('account.move', domain, ['amount_total', 'move_type'])
            df = SafetyUtils.safe_df(moves, ['amount_total', 'move_type'])
            
            if len(df) == 0:
                data = {"revenue": 0, "profit": 0, "expenses": 0, "profit_margin": 0}
            else:
                df['amount_total'] = pd.to_numeric(df['amount_total'], errors='coerce').fillna(0)
                revenue = df[df['move_type'] == 'out_invoice']['amount_total'].sum()
                expenses = df[df['move_type'] == 'in_invoice']['amount_total'].sum()
                profit = revenue - expenses
                data = {
                    "revenue": round(revenue, 2),
                    "expenses": round(expenses, 2),
                    "profit": round(profit, 2),
                    "profit_margin": round(SafetyUtils.calculate_percentage(profit, revenue), 2)
                }
            return AgentResult(data=data, timestamp=datetime.now(), success=True)
        except Exception as e:
            return AgentResult(data={}, timestamp=datetime.now(), success=False, error=str(e))

class CFOAgent:
    def execute(self, date_from=None, date_to=None) -> AgentResult:
        try:
            domain = [('state', '=', 'posted')]
            if date_from: domain.append(('date', '>=', date_from))
            if date_to: domain.append(('date', '<=', date_to))
            
            moves = odoo.read('account.move', domain, ['amount_total', 'move_type', 'payment_state'])
            df = SafetyUtils.safe_df(moves, ['amount_total', 'move_type', 'payment_state'])
            
            if len(df) == 0:
                data = {"cash_flow": 0, "accounts_receivable": 0, "accounts_payable": 0, "liquidity_status": "UNKNOWN"}
            else:
                df['amount_total'] = pd.to_numeric(df['amount_total'], errors='coerce').fillna(0)
                paid = df[df['payment_state'] == 'paid']
                cash_in = paid[paid['move_type'] == 'out_invoice']['amount_total'].sum()
                cash_out = paid[paid['move_type'] == 'in_invoice']['amount_total'].sum()
                
                ar = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()
                ap = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()
                
                cash_flow = cash_in - cash_out
                liquidity = "CRITICAL" if cash_flow < 0 and ar > 0 else "STRONG" if cash_flow > 0 else "ADEQUATE"
                
                data = {
                    "cash_flow": round(cash_flow, 2),
                    "accounts_receivable": round(ar, 2),
                    "accounts_payable": round(ap, 2),
                    "profit_cash_gap": round((ar - ap) - cash_flow, 2),
                    "cash_conversion_score": round(SafetyUtils.calculate_percentage(cash_in, cash_in+ar), 2),
                    "liquidity_status": liquidity
                }
            return AgentResult(data=data, timestamp=datetime.now(), success=True)
        except Exception as e:
            return AgentResult(data={}, timestamp=datetime.now(), success=False, error=str(e))

class CEOAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=Config.LLM_MODEL, temperature=0)
    
    def execute(self, metrics: BusinessMetrics, contradictions: Dict) -> AgentResult:
        try:
            context = {
                "financial": metrics.accounting,
                "cash": metrics.cfo,
                "sales": metrics.sales,
                "operations": {"delivery_rate": metrics.delivery.get('completion_rate'), 
                             "inventory_risk": metrics.inventory.get('stockout_risk_score')},
                "risks": contradictions
            }
            
            prompt = f"""You are the Group CEO. Write a Board Briefing based on this data:
{json.dumps(context, indent=2, default=str)}

STRUCTURE YOUR RESPONSE IN MARKDOWN:
### EXECUTIVE SUMMARY
[One compelling paragraph]

### CRITICAL FIRES TO FIGHT
[Bullet points of urgent issues]

### STRATEGIC DIRECTIVES
[Numbered actionable items for next 30 days]

### FINANCIAL VERDICT
[Honest assessment of cash vs profit]

Keep it professional, direct, and board-ready. Use bolding (**text**) for emphasis."""
            
            response = self.llm.invoke(prompt)
            return AgentResult(data={"full_brief": response.content}, timestamp=datetime.now(), success=True)
        except Exception as e:
            print(f"⚠️ CEO Agent Error: {e}")
            return AgentResult(data={"full_brief": "### ERROR\nAnalysis failed."}, timestamp=datetime.now(), success=False)

class ContradictionAgent:
    def execute(self, metrics: BusinessMetrics) -> AgentResult:
        contradictions = []
        warnings = []
        opportunities = []
        
        # Logic
        if metrics.sales.get('order_count', 0) > 0 and metrics.accounting.get('revenue', 0) == 0:
            contradictions.append("Sales Orders exist but Revenue is 0 (Invoicing Lag)")
        
        if metrics.accounting.get('profit', 0) > 0 and metrics.cfo.get('cash_flow', 0) <= 0:
            contradictions.append(f"Paper Profit (${metrics.accounting.get('profit')}) but Zero/Neg Cash Flow")
            
        if metrics.inventory.get('stockout_risk_score', 0) > 80:
            warnings.append("Critical Inventory Levels: High Stockout Risk")
            
        if metrics.cfo.get('accounts_receivable', 0) > metrics.sales.get('total_sales', 1) * 0.5:
            warnings.append("High Accounts Receivable Concentration")

        risk_score = (len(contradictions) * 25) + (len(warnings) * 10)
        risk_level = "CRITICAL" if risk_score > 50 else "HIGH" if risk_score > 30 else "LOW"
        
        return AgentResult(data={
            "contradictions": contradictions,
            "warnings": warnings,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "opportunities": opportunities
        }, timestamp=datetime.now(), success=True)

# ================================================================
# 6. ORCHESTRATOR
# ================================================================
class WorkflowOrchestrator:
    def __init__(self):
        self.sales = SalesAgent()
        self.delivery = DeliveryAgent()
        self.inventory = InventoryAgent()
        self.acct = AccountingAgent()
        self.cfo = CFOAgent()
        self.risk = ContradictionAgent()
        self.ceo = CEOAgent()
        self.graph = self._build_graph()
        
    def _build_graph(self):
        def collect(state):
            d_from, d_to = state.get('date_from'), state.get('date_to')
            metrics = BusinessMetrics(
                sales=self.sales.execute(d_from, d_to).data,
                delivery=self.delivery.execute(d_from, d_to).data,
                inventory=self.inventory.execute().data,
                accounting=self.acct.execute(d_from, d_to).data,
                cfo=self.cfo.execute(d_from, d_to).data,
                timestamp=datetime.now()
            )
            return {'metrics': metrics, 'date_from': d_from, 'date_to': d_to}

        def analyze(state):
            return {'contradictions': self.risk.execute(state['metrics']).data}

        def synthesize(state):
            return {'ceo_brief': self.ceo.execute(state['metrics'], state['contradictions']).data}

        g = StateGraph(GraphState)
        g.add_node("collect", collect)
        g.add_node("analyze", analyze)
        g.add_node("synthesize", synthesize)
        g.set_entry_point("collect")
        g.add_edge("collect", "analyze")
        g.add_edge("analyze", "synthesize")
        g.add_edge("synthesize", END)
        return g.compile()
    
    def execute(self, date_from=None, date_to=None):
        return self.graph.invoke({"date_from": date_from, "date_to": date_to})

# ================================================================
# 7. PROFESSIONAL PDF GENERATOR (REWRITTEN)
# ================================================================
class ProfessionalPDFGenerator:
    """
    Enhanced PDF Generator that parses Markdown and creates board-ready layouts.
    """
    def __init__(self):
        self.styles = getSampleStyleSheet()
        self._setup_custom_styles()
    
    def _setup_custom_styles(self):
        # Professional Color Palette
        self.color_primary = colors.HexColor('#1a237e')  # Deep Blue
        self.color_secondary = colors.HexColor('#283593')
        self.color_accent = colors.HexColor('#c62828')   # Alert Red
        self.color_grey = colors.HexColor('#455a64')
        self.color_success = colors.HexColor('#2e7d32')
        self.color_warning = colors.HexColor('#ef6c00')
        
        # Styles
        self.styles.add(ParagraphStyle(
            name='ReportTitle', parent=self.styles['Heading1'],
            fontSize=28, textColor=self.color_primary, alignment=TA_CENTER,
            spaceAfter=20, fontName='Helvetica-Bold'
        ))
        
        self.styles.add(ParagraphStyle(
            name='ReportSubtitle', parent=self.styles['Normal'],
            fontSize=14, textColor=self.color_grey, alignment=TA_CENTER,
            spaceAfter=40
        ))
        
        self.styles.add(ParagraphStyle(
            name='SectionHeader', parent=self.styles['Heading2'],
            fontSize=18, textColor=self.color_primary,
            spaceBefore=15, spaceAfter=10, borderPadding=5,
            borderColor=colors.lightgrey, borderWidth=0, borderBottomWidth=1
        ))
        
        self.styles.add(ParagraphStyle(
            name='MarkdownBold', parent=self.styles['Normal'],
            fontSize=10, leading=14
        ))

        self.styles.add(ParagraphStyle(
            name='MarkdownBullet', parent=self.styles['Normal'],
            fontSize=10, leading=14, leftIndent=20, bulletIndent=10,
            spaceAfter=3
        ))
    
    def _parse_markdown(self, text: str) -> List:
        """Parses Markdown text into ReportLab Paragraphs"""
        story_elements = []
        if not text: return story_elements
        
        lines = text.split('\n')
        
        for line in lines:
            line = line.strip()
            if not line:
                story_elements.append(Spacer(1, 5))
                continue
                
            # Parse Bold (**text**) -> <b>text</b>
            line = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', line)
            
            # Headers (###)
            if line.startswith('###'):
                clean_line = line.replace('###', '').strip()
                story_elements.append(Paragraph(clean_line, self.styles['SectionHeader']))
            
            # Bullets (- or *)
            elif line.startswith('- ') or line.startswith('* '):
                clean_line = line[1:].strip()
                story_elements.append(Paragraph(f"• {clean_line}", self.styles['MarkdownBullet']))
                
            # Numbered Lists (1. )
            elif re.match(r'^\d+\.', line):
                story_elements.append(Paragraph(line, self.styles['MarkdownBullet']))
                
            # Normal Text
            else:
                story_elements.append(Paragraph(line, self.styles['MarkdownBold']))
                
        return story_elements

    def _header_footer(self, canvas, doc):
        canvas.saveState()
        # Footer
        canvas.setFont('Helvetica', 8)
        canvas.setFillColor(colors.grey)
        canvas.drawString(inch, 0.75 * inch, f"ODIN Executive Suite | Generated: {datetime.now().strftime('%Y-%m-%d')}")
        canvas.drawRightString(doc.pagesize[0] - inch, 0.75 * inch, f"Page {doc.page}")
        canvas.restoreState()

    def _create_styled_table(self, data, col_widths, title_color=None):
        """Helper to create consistently styled tables"""
        if not title_color:
            title_color = self.color_primary
            
        t = Table(data, colWidths=col_widths)
        t.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), title_color),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('ALIGN', (0,0), (-1,-1), 'LEFT'),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('BOTTOMPADDING', (0,0), (-1,0), 12),
            ('GRID', (0,0), (-1,-1), 1, colors.lightgrey),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.whitesmoke, colors.white])
        ]))
        return t

    def _get_status_text(self, condition_good: bool, good_text="OK", bad_text="ALERT") -> str:
        """Returns colored Paragraph XML for status"""
        if condition_good:
            return f"<font color='green'>{good_text}</font>"
        return f"<font color='red'>{bad_text}</font>"

    def generate(self, data: Dict, period: str) -> str:
        filename = f"{Config.REPORT_DIR}/ODIN_Board_Report_{period}_{datetime.now().strftime('%Y%m%d_%H%M')}.pdf"
        doc = SimpleDocTemplate(filename, pagesize=letter, rightMargin=72, leftMargin=72, topMargin=72, bottomMargin=72)
        
        story = []
        metrics = data.get('metrics')
        contradictions = data.get('contradictions', {})
        ceo_brief = data.get('ceo_brief', {}).get('full_brief', '')
        
        # --- TITLE PAGE ---
        story.append(Spacer(1, 100))
        story.append(Paragraph("ODIN EXECUTIVE SUITE", self.styles['ReportTitle']))
        story.append(Paragraph(f"BOARD REPORT: {period}", self.styles['ReportSubtitle']))
        story.append(Paragraph("CONFIDENTIAL - BOARD USE ONLY", self.styles['ReportSubtitle']))
        story.append(PageBreak())
        
        # --- CEO BRIEF ---
        story.extend(self._parse_markdown(ceo_brief))
        story.append(PageBreak())
        
        # --- OPERATIONAL INTELLIGENCE ---
        story.append(Paragraph("OPERATIONAL INTELLIGENCE", self.styles['SectionHeader']))
        
        if metrics:
            # 1. SALES
            story.append(Paragraph("Sales Performance", self.styles['MarkdownBold']))
            story.append(Spacer(1, 5))
            
            sales_status = Paragraph(self._get_status_text(metrics.sales.get('total_sales', 0) > 0, "HEALTHY", "LOW VOLUME"), self.styles['MarkdownBold'])
            
            sales_data = [
                ['METRIC', 'VALUE', 'STATUS'],
                ['Total Sales', f"${metrics.sales.get('total_sales', 0):,.2f}", sales_status],
                ['Order Count', f"{metrics.sales.get('order_count', 0)}", ''],
                ['Avg Order Value', f"${metrics.sales.get('average_order_value', 0):,.2f}", ''],
                ['Conversion Rate', f"{metrics.sales.get('conversion_rate', 0)}%", ''],
                ['Draft Orders', f"{metrics.sales.get('draft_orders', 0)}", ''],
            ]
            story.append(self._create_styled_table(sales_data, [200, 150, 80], self.color_secondary))
            story.append(Spacer(1, 15))

            # 2. DELIVERY
            story.append(Paragraph("Delivery Logistics", self.styles['MarkdownBold']))
            story.append(Spacer(1, 5))
            
            del_status = Paragraph(self._get_status_text(metrics.delivery.get('completion_rate', 0) >= 80, "ON TRACK", "DELAYED"), self.styles['MarkdownBold'])
            
            del_data = [
                ['METRIC', 'VALUE', 'STATUS'],
                ['Completion Rate', f"{metrics.delivery.get('completion_rate', 0)}%", del_status],
                ['Total Deliveries', f"{metrics.delivery.get('total_deliveries', 0)}", ''],
                ['Pending', f"{metrics.delivery.get('pending', 0)}", ''],
                ['Completed', f"{metrics.delivery.get('completed', 0)}", ''],
                ['Efficiency Score', f"{metrics.delivery.get('efficiency_score', 0)}", ''],
            ]
            story.append(self._create_styled_table(del_data, [200, 150, 80], self.color_grey))
            story.append(Spacer(1, 15))

            # 3. INVENTORY
            story.append(Paragraph("Inventory Health", self.styles['MarkdownBold']))
            story.append(Spacer(1, 5))
            
            inv_good = 'CRITICAL' not in metrics.inventory.get('health_status', '')
            inv_status_txt = Paragraph(self._get_status_text(inv_good, "STABLE", "RISK"), self.styles['MarkdownBold'])
            zero_status = Paragraph(self._get_status_text(metrics.inventory.get('zero_negative_items', 0) == 0, "NONE", "ACTION"), self.styles['MarkdownBold'])

            inv_data = [
                ['METRIC', 'VALUE', 'STATUS'],
                ['Health Status', metrics.inventory.get('health_status', 'UNKNOWN'), inv_status_txt],
                ['Stockout Risk', f"{metrics.inventory.get('stockout_risk_score', 0)}%", ''],
                ['Available Units', f"{metrics.inventory.get('available_units', 0):,.0f}", ''],
                ['Zero/Negative Items', f"{metrics.inventory.get('zero_negative_items', 0)}", zero_status],
            ]
            story.append(self._create_styled_table(inv_data, [200, 150, 80], colors.purple))
            story.append(PageBreak())
        
        # --- FINANCIAL DEEP DIVE ---
        story.append(Paragraph("FINANCIAL DEEP DIVE", self.styles['SectionHeader']))
        
        if metrics:
            # Profitability
            story.append(Paragraph("Profitability & Margins", self.styles['MarkdownBold']))
            story.append(Spacer(1, 5))
            
            profit_status = Paragraph(self._get_status_text(metrics.accounting.get('profit', 0) > 0, "PROFIT", "LOSS"), self.styles['MarkdownBold'])
            
            fin_data = [
                ['METRIC', 'VALUE', 'STATUS'],
                ['Total Revenue', f"${metrics.accounting.get('revenue', 0):,.2f}", ''],
                ['Total Expenses', f"${metrics.accounting.get('expenses', 0):,.2f}", ''],
                ['Net Profit', f"${metrics.accounting.get('profit', 0):,.2f}", profit_status],
                ['Profit Margin', f"{metrics.accounting.get('profit_margin', 0)}%", ''],
            ]
            story.append(self._create_styled_table(fin_data, [200, 150, 80], self.color_primary))
            story.append(Spacer(1, 15))

            # Liquidity
            story.append(Paragraph("Liquidity & Cash Flow", self.styles['MarkdownBold']))
            story.append(Spacer(1, 5))
            
            cash_status = Paragraph(self._get_status_text(metrics.cfo.get('cash_flow', 0) > 0, "POSITIVE", "NEGATIVE"), self.styles['MarkdownBold'])
            
            cash_data = [
                ['METRIC', 'VALUE', 'STATUS'],
                ['Operating Cash Flow', f"${metrics.cfo.get('cash_flow', 0):,.2f}", cash_status],
                ['Accounts Receivable', f"${metrics.cfo.get('accounts_receivable', 0):,.2f}", ''],
                ['Accounts Payable', f"${metrics.cfo.get('accounts_payable', 0):,.2f}", ''],
                ['Profit-Cash Gap', f"${metrics.cfo.get('profit_cash_gap', 0):,.2f}", ''],
                ['Liquidity Status', metrics.cfo.get('liquidity_status', 'UNKNOWN'), ''],
            ]
            story.append(self._create_styled_table(cash_data, [200, 150, 80], self.color_primary))

        # --- RISK ANALYSIS ---
        story.append(Spacer(1, 20))
        story.append(Paragraph("RISK MONITOR", self.styles['SectionHeader']))
        
        if contradictions:
            risk_level = contradictions.get('risk_level', 'LOW')
            color = colors.red if risk_level == 'CRITICAL' else colors.orange if risk_level == 'HIGH' else colors.green
            
            story.append(Paragraph(f"<b>Overall Risk Level:</b> <font color='{color}'>{risk_level}</font>", self.styles['MarkdownBold']))
            story.append(Spacer(1, 10))
            
            # Issues Table
            issues = contradictions.get('contradictions', [])
            warnings = contradictions.get('warnings', [])
            
            if issues:
                story.append(Paragraph("<b>CRITICAL ISSUES (Immediate Action Required):</b>", self.styles['MarkdownBold']))
                for issue in issues:
                    story.append(Paragraph(f"• {issue}", self.styles['MarkdownBullet']))
                story.append(Spacer(1, 10))

            if warnings:
                story.append(Paragraph("<b>WARNINGS (Monitor Closely):</b>", self.styles['MarkdownBold']))
                for warn in warnings:
                    story.append(Paragraph(f"• {warn}", self.styles['MarkdownBullet']))
        
        # Build
        doc.build(story, onFirstPage=self._header_footer, onLaterPages=self._header_footer)
        return filename

# ================================================================
# 8. MAIN EXECUTION
# ================================================================
if __name__ == "__main__":
    print("🚀 ODIN Executive Suite v4.2 - Starting...")
    
    # Init
    odin = WorkflowOrchestrator()
    pdf_gen = ProfessionalPDFGenerator()
    
    # Define Dates (Last 90 Days)
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
    
    print(f"📊 Analyzing data from {start_date} to {end_date}...")
    
    # Run Workflow
    result = odin.execute(start_date, end_date)
    
    # Generate PDF
    print("📝 Generating Professional PDF Report...")
    try:
        report_path = pdf_gen.generate(result, "2025-Q4")
        print(f"\n✅ SUCCESS: Report generated at:\n   {report_path}")
    except Exception as e:
        print(f"\n❌ ERROR: Could not generate PDF.\n   {e}")

✅ Connected to Odoo
🚀 ODIN Executive Suite v4.2 - Starting...
📊 Analyzing data from 2025-09-18 to 2025-12-17...
📝 Generating Professional PDF Report...

✅ SUCCESS: Report generated at:
   ./odin_reports/ODIN_Board_Report_2025-Q4_20251217_0842.pdf
